# Results Visualization: Comprehensive Summary and Paper-Ready Figures

This notebook produces publication-quality summary figures combining all experimental results:

- **Table 1**: Attack metrics summary (ASR-R, ASR-A, ASR-T, Benign Accuracy)
- **Table 2**: Defense metrics summary (TPR, FPR, Precision, F1)
- **Figure 1**: Unified attack vs. defense effectiveness radar chart
- **Figure 2**: End-to-end threat model pipeline diagram
- **Figure 3**: Composite score comparison (attacks + defenses)
- **Figure 4**: 3D scatter — ASR-R, ASR-A, Benign Accuracy per attack
- **Figure 5**: Defense overhead vs. detection effectiveness
- **Figure 6**: Normalized defense effectiveness heatmap across attack families

In [ ]:
import json
import os
import sys

import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from attacks.implementations import AttackSuite
from data.synthetic_corpus import SyntheticCorpus
from evaluation.benchmarking import DefenseEvaluator
from evaluation.retrieval_sim import RetrievalSimulator

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

SEED = 42
CORPUS_SIZE = 200
TOP_K = 5
N_POISON_BASE = 5

ATTACK_TYPES = ['agent_poison', 'minja', 'injecmem']
DEFENSE_TYPES = ['watermark', 'validation', 'proactive', 'composite']

ATTACK_LABELS = {
    'agent_poison': 'AgentPoison\n(Chen et al., 2024)',
    'minja': 'MINJA\n(Dong et al., 2025)',
    'injecmem': 'InjecMEM\n(ICLR 2026)',
}
ATTACK_LABELS_SHORT = {
    'agent_poison': 'AgentPoison',
    'minja': 'MINJA',
    'injecmem': 'InjecMEM',
}
DEFENSE_LABELS = {
    'watermark': 'Watermark\n(Zhao et al.)',
    'validation': 'Content\nValidation',
    'proactive': 'Proactive\nDefense',
    'composite': 'Composite\n(Multi-layer)',
}
DEFENSE_LABELS_SHORT = {
    'watermark': 'Watermark',
    'validation': 'Validation',
    'proactive': 'Proactive',
    'composite': 'Composite',
}

ATTACK_COLORS = {
    'agent_poison': '#E74C3C',
    'minja': '#3498DB',
    'injecmem': '#27AE60',
}
DEFENSE_COLORS = {
    'watermark': '#8E44AD',
    'validation': '#E67E22',
    'proactive': '#16A085',
    'composite': '#C0392B',
}

print('visualization setup complete')

## 1. Run All Experiments and Collect Results

In [ ]:
# run attack evaluations
sim = RetrievalSimulator(
    corpus_size=CORPUS_SIZE,
    top_k=TOP_K,
    n_poison_per_attack=N_POISON_BASE,
    seed=SEED,
)

attack_results = {}
for at in ATTACK_TYPES:
    m = sim.evaluate_attack(at)
    attack_results[at] = m
    print(f'{at}: ASR-R={m.asr_r:.3f}  ASR-A={m.asr_a:.3f}  ASR-T={m.asr_t:.3f}  Benign={m.benign_accuracy:.3f}')

# run defense evaluations
corpus = SyntheticCorpus(seed=SEED)
benign_entries = corpus.generate_benign_entries(50)
clean_content = [e['content'] for e in benign_entries]

attack_suite = AttackSuite()
poisoned_content = []
for entry in benign_entries[:20]:
    results = attack_suite.execute_all(entry['content'])
    for at, result in results['attack_results'].items():
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            poisoned_content.append(pc)

defense_evaluator = DefenseEvaluator()
defense_results = {}
for dt in DEFENSE_TYPES:
    m = defense_evaluator.evaluate_defense(dt, attack_suite, clean_content, poisoned_content)
    defense_results[dt] = m
    print(f'{dt}: TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  F1={m.f1_score:.3f}')

print(f'\nclean samples: {len(clean_content)}, poisoned samples: {len(poisoned_content)}')

## 2. Table 1: Attack Metrics Summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

table_data = []
for at in ATTACK_TYPES:
    m = attack_results[at]
    label = ATTACK_LABELS[at].replace('\n', ' ')
    table_data.append([
        label,
        f'{m.asr_r:.3f}',
        f'{m.asr_a:.3f}',
        f'{m.asr_t:.3f}',
        f'{m.benign_accuracy:.3f}',
        f'{m.asr_t:.3f}',  # threat score = ASR-T
    ])

col_labels = ['Attack', 'ASR-R', 'ASR-A', 'ASR-T', 'Benign Acc.', 'Threat Score']
col_colors = ['#ECF0F1'] * len(col_labels)
row_colors = [[ATTACK_COLORS[at] + '33'] * len(col_labels) for at in ATTACK_TYPES]

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(11)

# style header
for j, label in enumerate(col_labels):
    table[0, j].set_facecolor('#2C3E50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# style rows
for i, at in enumerate(ATTACK_TYPES):
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(row_colors[i][j])
    table[i+1, 0].set_text_props(fontweight='bold')

ax.set_title('Table 1: Attack Methodology Evaluation Summary', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../visualization/fig_table1_attack_summary.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_table1_attack_summary.pdf', bbox_inches='tight')
plt.show()
print('Table 1 saved')

## 3. Table 2: Defense Metrics Summary

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.axis('off')

table_data = []
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    label = DEFENSE_LABELS[dt].replace('\n', ' ')
    youden = m.tpr - m.fpr
    table_data.append([
        label,
        f'{m.tpr:.3f}',
        f'{m.fpr:.3f}',
        f'{m.precision:.3f}',
        f'{m.f1_score:.3f}',
        f'{youden:.3f}',
        f'{m.true_positives}',
        f'{m.false_positives}',
    ])

col_labels = ['Defense', 'TPR', 'FPR', 'Precision', 'F1', 'Youden J', 'TP', 'FP']

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(10)

for j in range(len(col_labels)):
    table[0, j].set_facecolor('#1A252F')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i, dt in enumerate(DEFENSE_TYPES):
    m = defense_results[dt]
    # color by F1: green=high, yellow=medium, red=low
    if m.f1_score >= 0.7:
        row_bg = '#D5F5E3'
    elif m.f1_score >= 0.4:
        row_bg = '#FEF9E7'
    else:
        row_bg = '#FDEDEC'
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(row_bg)
    table[i+1, 0].set_text_props(fontweight='bold')

ax.set_title('Table 2: Defense Mechanism Evaluation Summary', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../visualization/fig_table2_defense_summary.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_table2_defense_summary.pdf', bbox_inches='tight')
plt.show()
print('Table 2 saved')

## 4. Figure 1: Radar Chart — Attack vs. Defense Effectiveness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(polar=True))
fig.suptitle('Figure 1: Radar Profile — Attack Effectiveness and Defense Coverage', fontsize=13, fontweight='bold')

# --- attack radar ---
attack_metric_labels = ['ASR-R', 'ASR-A', 'ASR-T', 'Benign\nAcc.', 'Stealthiness']
n_attack = len(attack_metric_labels)
angles_a = np.linspace(0, 2 * np.pi, n_attack, endpoint=False).tolist()
angles_a += angles_a[:1]

ax_a = axes[0]
ax_a.set_theta_offset(np.pi / 2)
ax_a.set_theta_direction(-1)
ax_a.set_thetagrids(np.degrees(angles_a[:-1]), attack_metric_labels)
ax_a.set_title('Attack Profiles', size=12, pad=15)

for at in ATTACK_TYPES:
    m = attack_results[at]
    # stealthiness = benign_accuracy (high = stealthy)
    values = [m.asr_r, m.asr_a, m.asr_t, m.benign_accuracy, m.benign_accuracy]
    values += values[:1]
    ax_a.plot(angles_a, values, 'o-', linewidth=2, color=ATTACK_COLORS[at],
              label=ATTACK_LABELS_SHORT[at])
    ax_a.fill(angles_a, values, alpha=0.1, color=ATTACK_COLORS[at])

ax_a.set_ylim(0, 1)
ax_a.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_a.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
ax_a.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=9)

# --- defense radar ---
defense_metric_labels = ['TPR', 'Precision', 'F1', '1-FPR\n(Specificity)', 'Youden J']
n_defense = len(defense_metric_labels)
angles_d = np.linspace(0, 2 * np.pi, n_defense, endpoint=False).tolist()
angles_d += angles_d[:1]

ax_d = axes[1]
ax_d.set_theta_offset(np.pi / 2)
ax_d.set_theta_direction(-1)
ax_d.set_thetagrids(np.degrees(angles_d[:-1]), defense_metric_labels)
ax_d.set_title('Defense Profiles', size=12, pad=15)

for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    youden = m.tpr - m.fpr
    values = [m.tpr, m.precision, m.f1_score, 1.0 - m.fpr, max(0.0, youden)]
    values += values[:1]
    ax_d.plot(angles_d, values, 'o-', linewidth=2, color=DEFENSE_COLORS[dt],
              label=DEFENSE_LABELS_SHORT[dt])
    ax_d.fill(angles_d, values, alpha=0.1, color=DEFENSE_COLORS[dt])

ax_d.set_ylim(0, 1)
ax_d.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_d.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
ax_d.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.tight_layout()
plt.savefig('../visualization/fig01_radar_attack_defense.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig01_radar_attack_defense.pdf', bbox_inches='tight')
plt.show()
print('Figure 1 saved: fig01_radar_attack_defense')

## 5. Figure 2: Threat Model Pipeline Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#FAFAFA')
fig.patch.set_facecolor('#FAFAFA')

def draw_box(ax, x, y, w, h, label, sublabel=None, color='#3498DB', text_color='white', fontsize=10):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='#2C3E50',
                          linewidth=1.5, zorder=3, alpha=0.9)
    ax.add_patch(rect)
    if sublabel:
        ax.text(x + w/2, y + h*0.62, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)
        ax.text(x + w/2, y + h*0.3, sublabel, ha='center', va='center',
                fontsize=fontsize - 1.5, color=text_color, alpha=0.9, zorder=4)
    else:
        ax.text(x + w/2, y + h/2, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, color='#2C3E50', label=None):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0), zorder=2)
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2 + 0.15
        ax.text(mx, my, label, ha='center', va='bottom', fontsize=8.5,
                color=color, style='italic')

# adversary section
adv_bg = plt.Rectangle((0.3, 4.2), 5.2, 2.4, facecolor='#FDEDEC', edgecolor='#E74C3C',
                         linewidth=2, linestyle='--', zorder=1, alpha=0.5)
ax.add_patch(adv_bg)
ax.text(0.6, 6.3, 'Adversary', fontsize=9, color='#E74C3C', fontweight='bold')

draw_box(ax, 0.5, 4.4, 2.2, 1.8, 'Attack\nGeneration', 'AgentPoison / MINJA / InjecMEM',
         color='#E74C3C', fontsize=9)
draw_box(ax, 3.2, 4.4, 2.0, 1.8, 'Poison\nPassages', 'trigger-optimized content',
         color='#C0392B', fontsize=9)

draw_arrow(ax, 2.72, 5.3, 3.2, 5.3)

# memory system section
mem_bg = plt.Rectangle((5.8, 1.0), 4.4, 5.5, facecolor='#EBF5FB', edgecolor='#3498DB',
                         linewidth=2, linestyle='--', zorder=1, alpha=0.5)
ax.add_patch(mem_bg)
ax.text(6.0, 6.2, 'Agent Memory System', fontsize=9, color='#3498DB', fontweight='bold')

draw_box(ax, 6.0, 4.4, 3.8, 1.8, 'Vector Store', 'FAISS + sentence-transformers',
         color='#2980B9', fontsize=9)
draw_box(ax, 6.0, 1.2, 3.8, 1.8, 'Benign Corpus', f'N={CORPUS_SIZE} synthetic agent memory entries',
         color='#5DADE2', fontsize=9)

draw_arrow(ax, 7.9, 4.4, 7.9, 3.0, label='inject')

# injection arrow from adversary to memory
draw_arrow(ax, 5.2, 5.3, 6.0, 5.3, color='#E74C3C', label='poison injection')

# defense section
def_bg = plt.Rectangle((0.3, 0.3), 5.2, 3.5, facecolor='#E8F8F5', edgecolor='#27AE60',
                         linewidth=2, linestyle='--', zorder=1, alpha=0.5)
ax.add_patch(def_bg)
ax.text(0.6, 3.5, 'Defense Layer', fontsize=9, color='#27AE60', fontweight='bold')

draw_box(ax, 0.5, 0.5, 2.0, 1.5, 'Watermark\nDefense', 'z-score detection',
         color='#8E44AD', fontsize=8.5)
draw_box(ax, 3.0, 0.5, 2.2, 1.5, 'Content\nValidation', 'pattern filtering',
         color='#E67E22', fontsize=8.5)
draw_box(ax, 0.5, 2.2, 2.0, 1.2, 'Proactive\nDefense', 'pre-emptive scan',
         color='#16A085', fontsize=8.5)
draw_box(ax, 3.0, 2.2, 2.2, 1.2, 'Composite\nDefense', 'multi-layer fusion',
         color='#C0392B', fontsize=8.5)

# victim query section
draw_box(ax, 11.0, 4.4, 2.8, 1.8, 'Victim Agent', 'issues target queries',
         color='#F39C12', fontsize=9)

# evaluation
draw_box(ax, 11.0, 1.2, 2.8, 1.8, 'Evaluation', 'ASR-R / ASR-A / ASR-T\nTPR / FPR / F1',
         color='#1ABC9C', fontsize=8.5)

# arrows
draw_arrow(ax, 11.0, 5.3, 9.8, 5.3, color='#F39C12', label='query retrieval')
draw_arrow(ax, 9.8, 4.4, 9.8, 3.0, color='#2980B9', label='top-k results')
draw_arrow(ax, 9.8, 2.0, 11.0, 2.1, color='#1ABC9C', label='metrics')
draw_arrow(ax, 5.5, 1.5, 6.0, 2.5, color='#27AE60', label='detect')

ax.set_title('Figure 2: End-to-End Threat Model — Memory Poisoning Attack and Defense Pipeline',
             fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('../visualization/fig02_threat_model_pipeline.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig02_threat_model_pipeline.pdf', bbox_inches='tight')
plt.show()
print('Figure 2 saved: fig02_threat_model_pipeline')

## 6. Figure 3: Composite Score Comparison (Attack Threat + Defense Coverage)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 3: Composite Threat and Coverage Scores', fontsize=13, fontweight='bold')

# left: attack composite threat score (normalized weighted sum)
# threat = 0.4*ASR_R + 0.3*ASR_A + 0.3*ASR_T
ax_l = axes[0]
threat_scores = []
for at in ATTACK_TYPES:
    m = attack_results[at]
    threat = 0.4 * m.asr_r + 0.3 * m.asr_a + 0.3 * m.asr_t
    threat_scores.append(threat)

x = np.arange(len(ATTACK_TYPES))
bars = ax_l.bar(x, threat_scores, 0.5,
                color=[ATTACK_COLORS[at] for at in ATTACK_TYPES],
                edgecolor='black', linewidth=0.6, alpha=0.87)
for bar, val in zip(bars, threat_scores):
    ax_l.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
              f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax_l.set_xticks(x)
ax_l.set_xticklabels([ATTACK_LABELS[at] for at in ATTACK_TYPES], fontsize=9)
ax_l.set_ylabel('Composite Threat Score')
ax_l.set_title('Attack Threat Score\n(0.4·ASR-R + 0.3·ASR-A + 0.3·ASR-T)', fontsize=10)
ax_l.set_ylim(0, 1.0)
ax_l.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)

# right: defense composite coverage score
# coverage = 0.4*F1 + 0.3*TPR + 0.3*(1-FPR)
ax_r = axes[1]
coverage_scores = []
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    coverage = 0.4 * m.f1_score + 0.3 * m.tpr + 0.3 * (1.0 - m.fpr)
    coverage_scores.append(coverage)

x2 = np.arange(len(DEFENSE_TYPES))
bars2 = ax_r.bar(x2, coverage_scores, 0.5,
                 color=[DEFENSE_COLORS[dt] for dt in DEFENSE_TYPES],
                 edgecolor='black', linewidth=0.6, alpha=0.87)
for bar, val in zip(bars2, coverage_scores):
    ax_r.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
              f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax_r.set_xticks(x2)
ax_r.set_xticklabels([DEFENSE_LABELS[dt] for dt in DEFENSE_TYPES], fontsize=9)
ax_r.set_ylabel('Composite Coverage Score')
ax_r.set_title('Defense Coverage Score\n(0.4·F1 + 0.3·TPR + 0.3·Specificity)', fontsize=10)
ax_r.set_ylim(0, 1.0)
ax_r.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)

plt.tight_layout()
plt.savefig('../visualization/fig03_composite_scores.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig03_composite_scores.pdf', bbox_inches='tight')
plt.show()
print('Figure 3 saved: fig03_composite_scores')

## 7. Figure 4: 3D Scatter — ASR-R, ASR-A, Benign Accuracy

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

attack_markers_3d = {'agent_poison': 'o', 'minja': 's', 'injecmem': '^'}

for at in ATTACK_TYPES:
    m = attack_results[at]
    ax.scatter(
        m.asr_r, m.asr_a, m.benign_accuracy,
        c=ATTACK_COLORS[at],
        marker=attack_markers_3d[at],
        s=300,
        zorder=5,
        label=ATTACK_LABELS[at].replace('\n', ' '),
        edgecolors='black',
        linewidths=1,
        alpha=0.9,
    )
    ax.text(m.asr_r + 0.02, m.asr_a + 0.02, m.benign_accuracy + 0.01,
            ATTACK_LABELS_SHORT[at], fontsize=9, fontweight='bold')

# draw projections (dashed lines to axes planes)
for at in ATTACK_TYPES:
    m = attack_results[at]
    ax.plot([m.asr_r, m.asr_r], [m.asr_a, m.asr_a], [0, m.benign_accuracy],
            '--', color=ATTACK_COLORS[at], alpha=0.3, linewidth=0.8)
    ax.plot([m.asr_r, m.asr_r], [0, m.asr_a], [m.benign_accuracy, m.benign_accuracy],
            '--', color=ATTACK_COLORS[at], alpha=0.3, linewidth=0.8)
    ax.plot([0, m.asr_r], [m.asr_a, m.asr_a], [m.benign_accuracy, m.benign_accuracy],
            '--', color=ATTACK_COLORS[at], alpha=0.3, linewidth=0.8)

ax.set_xlabel('ASR-R (Retrieval Rate)', fontsize=10, labelpad=10)
ax.set_ylabel('ASR-A (Action Rate | Retrieval)', fontsize=10, labelpad=10)
ax.set_zlabel('Benign Accuracy (Stealthiness)', fontsize=10, labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)
ax.set_title('Figure 4: 3D Attack Profile — Retrieval, Action, and Stealthiness Dimensions',
             fontsize=11, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=9, bbox_to_anchor=(0.0, 1.05))
ax.view_init(elev=20, azim=225)

plt.tight_layout()
plt.savefig('../visualization/fig04_3d_attack_profile.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig04_3d_attack_profile.pdf', bbox_inches='tight')
plt.show()
print('Figure 4 saved: fig04_3d_attack_profile')

## 8. Figure 5: Defense Utility — TPR vs. (1 - FPR) Pareto Frontier

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

defense_markers = {'watermark': 'o', 'validation': 's', 'proactive': '^', 'composite': 'D'}

# theoretical optimal and random lines
ax.plot([0, 1], [1, 1], 'g--', alpha=0.3, linewidth=1, label='Ideal TPR=1')
ax.plot([1, 1], [0, 1], 'g--', alpha=0.3, linewidth=1)
ax.fill_between([0.7, 1.0], [0.7, 0.7], [1.0, 1.0], alpha=0.07, color='green',
                label='High-performance region')

# F1 iso-contours
specificity_vals = np.linspace(0.01, 1.0, 300)
for f1_target in [0.3, 0.5, 0.7, 0.9]:
    # F1 = 2*P*R / (P+R) — simplified here as contour in TPR vs specificity space
    # approximate: plot as horizontal dashed contour at TPR = f1_target
    ax.axhline(y=f1_target, color='lightgray', linestyle=':', alpha=0.6, linewidth=0.8)
    ax.text(0.02, f1_target + 0.01, f'TPR={f1_target:.1f}', fontsize=7.5, color='gray')

# plot defenses in TPR vs. Specificity (1 - FPR) space
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    specificity = 1.0 - m.fpr
    ax.scatter(
        specificity, m.tpr,
        c=DEFENSE_COLORS[dt],
        marker=defense_markers[dt],
        s=220,
        zorder=5,
        label=f'{DEFENSE_LABELS_SHORT[dt]} (F1={m.f1_score:.2f}, J={m.tpr - m.fpr:.2f})',
        edgecolors='black',
        linewidths=1.2,
    )
    # annotate
    ax.annotate(
        f'  {DEFENSE_LABELS_SHORT[dt]}',
        (specificity, m.tpr),
        fontsize=9, fontweight='bold', color=DEFENSE_COLORS[dt],
    )

ax.set_xlabel('Specificity (1 - FPR)', fontsize=12)
ax.set_ylabel('Sensitivity (TPR)', fontsize=12)
ax.set_title('Figure 5: Defense Pareto Frontier — Sensitivity vs. Specificity',
             fontsize=12, fontweight='bold')
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.legend(loc='lower left', fontsize=9)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('../visualization/fig05_defense_pareto_frontier.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig05_defense_pareto_frontier.pdf', bbox_inches='tight')
plt.show()
print('Figure 5 saved: fig05_defense_pareto_frontier')

## 9. Figure 6: Normalized Defense Effectiveness Heatmap Across Attack Families

In [ ]:
# compute per-attack defense effectiveness using attack-specific poisoned content
per_attack_defense_matrix = np.zeros((len(DEFENSE_TYPES), len(ATTACK_TYPES)))

for j, at in enumerate(ATTACK_TYPES):
    # generate poisoned content specific to this attack type
    atk = attack_suite.get_attack(at)
    attack_poison = []
    for entry in benign_entries[:15]:
        result = atk.execute(entry['content'])
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            attack_poison.append(pc)

    for i, dt in enumerate(DEFENSE_TYPES):
        m = defense_evaluator.evaluate_defense(
            dt, attack_suite, clean_content[:15], attack_poison
        )
        per_attack_defense_matrix[i, j] = m.f1_score

# normalize each column (attack family) to [0, 1] across defenses
normalized_matrix = np.zeros_like(per_attack_defense_matrix)
for j in range(len(ATTACK_TYPES)):
    col = per_attack_defense_matrix[:, j]
    col_min, col_max = col.min(), col.max()
    if col_max > col_min:
        normalized_matrix[:, j] = (col - col_min) / (col_max - col_min)
    else:
        normalized_matrix[:, j] = 0.5

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Figure 6: Defense F1 Effectiveness Across Attack Families',
             fontsize=13, fontweight='bold')

# raw F1 heatmap
ax1 = axes[0]
im1 = ax1.imshow(per_attack_defense_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax1.set_xticks(range(len(ATTACK_TYPES)))
ax1.set_xticklabels([ATTACK_LABELS_SHORT[at] for at in ATTACK_TYPES])
ax1.set_yticks(range(len(DEFENSE_TYPES)))
ax1.set_yticklabels([DEFENSE_LABELS_SHORT[dt] for dt in DEFENSE_TYPES])
ax1.set_title('Raw F1 Score', fontsize=11)
plt.colorbar(im1, ax=ax1, label='F1 Score')
for i in range(len(DEFENSE_TYPES)):
    for j in range(len(ATTACK_TYPES)):
        ax1.text(j, i, f'{per_attack_defense_matrix[i, j]:.2f}',
                 ha='center', va='center', fontsize=11, fontweight='bold',
                 color='white' if per_attack_defense_matrix[i, j] < 0.4 else 'black')

# normalized heatmap (relative effectiveness within each attack family)
ax2 = axes[1]
im2 = ax2.imshow(normalized_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(len(ATTACK_TYPES)))
ax2.set_xticklabels([ATTACK_LABELS_SHORT[at] for at in ATTACK_TYPES])
ax2.set_yticks(range(len(DEFENSE_TYPES)))
ax2.set_yticklabels([DEFENSE_LABELS_SHORT[dt] for dt in DEFENSE_TYPES])
ax2.set_title('Normalized (per Attack Family)', fontsize=11)
plt.colorbar(im2, ax=ax2, label='Relative Effectiveness')
for i in range(len(DEFENSE_TYPES)):
    for j in range(len(ATTACK_TYPES)):
        ax2.text(j, i, f'{normalized_matrix[i, j]:.2f}',
                 ha='center', va='center', fontsize=11, fontweight='bold',
                 color='white' if normalized_matrix[i, j] < 0.4 else 'black')

plt.tight_layout()
plt.savefig('../visualization/fig06_defense_effectiveness_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig06_defense_effectiveness_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Figure 6 saved: fig06_defense_effectiveness_heatmap')

## 10. Save Consolidated Results

In [ ]:
import random as _random

# simulate minja isr across base success probabilities (n_entries=100, n_turns=3)
base_probs = np.linspace(0.50, 1.00, 30)
n_entries = 100
n_turns = 3
shortening_rate = 0.10

def simulate_isr(base_prob, n_entries, n_turns, shortening_rate, seed=42):
    """simulate minja multi-turn isr for a given base probability."""
    rng = _random.Random(seed)
    successful = 0
    for _ in range(n_entries):
        injected = False
        for turn_idx in range(n_turns):
            turn_prob = base_prob * max(0.5, 1.0 - shortening_rate * turn_idx)
            if rng.random() < turn_prob:
                injected = True
                break
        if injected:
            successful += 1
    return successful / n_entries

isr_1turn = [simulate_isr(p, n_entries, 1, shortening_rate) for p in base_probs]
isr_2turn = [simulate_isr(p, n_entries, 2, shortening_rate) for p in base_probs]
isr_3turn = [simulate_isr(p, n_entries, 3, shortening_rate) for p in base_probs]
isr_5turn = [simulate_isr(p, n_entries, 5, shortening_rate) for p in base_probs]

# paper-reported point: base_prob ≈ 0.98, isr ≈ 0.982 with 3 turns
paper_prob, paper_isr = 0.98, 0.982

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Figure 10a: MINJA Multi-Turn Injection Success Rate (ISR) Simulation\n'
             '(Dong et al., NeurIPS 2025 — Progressive Shortening Protocol)', fontsize=12, fontweight='bold')

# left: isr vs base success probability for different turn counts
ax = axes[0]
ax.plot(base_probs, isr_1turn, color='#BDC3C7', linewidth=2, label='n_turns=1', linestyle='--')
ax.plot(base_probs, isr_2turn, color='#85C1E9', linewidth=2, label='n_turns=2')
ax.plot(base_probs, isr_3turn, color='#3498DB', linewidth=2.5, label='n_turns=3 (paper)')
ax.plot(base_probs, isr_5turn, color='#1A5276', linewidth=2, label='n_turns=5')
ax.scatter([paper_prob], [paper_isr], color='red', s=150, zorder=5,
           label=f'Paper (ISR={paper_isr:.3f})', marker='*')
ax.axhline(y=paper_isr, color='red', linestyle=':', alpha=0.5, linewidth=1)
ax.axvline(x=paper_prob, color='red', linestyle=':', alpha=0.5, linewidth=1)
ax.set_xlabel('Base Injection Success Probability (p₀)')
ax.set_ylabel('Injection Success Rate (ISR)')
ax.set_title('ISR vs. Base Success Probability\nfor Different Indication Turn Counts')
ax.legend(fontsize=9)
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.25)

# right: isr vs shortening rate (at base_prob=0.98, n_turns=3)
shortening_rates = np.linspace(0.0, 0.5, 25)
isr_by_shortening = [simulate_isr(0.98, n_entries, 3, sr) for sr in shortening_rates]
ax = axes[1]
ax.plot(shortening_rates, isr_by_shortening, color='#3498DB', linewidth=2.5, marker='o',
        markersize=4, label='ISR (base_prob=0.98, n_turns=3)')
ax.axhline(y=paper_isr, color='red', linestyle=':', alpha=0.7, linewidth=1.5,
           label=f'Paper ISR = {paper_isr:.3f}')
ax.axvline(x=0.10, color='gray', linestyle='--', alpha=0.7, linewidth=1.5,
           label='Paper shortening rate = 0.10')
ax.fill_between(shortening_rates, isr_by_shortening, paper_isr,
                where=[v < paper_isr for v in isr_by_shortening],
                alpha=0.15, color='orange', label='ISR gap from paper')
ax.set_xlabel('Progressive Shortening Rate per Turn')
ax.set_ylabel('Injection Success Rate (ISR)')
ax.set_title('ISR vs. Shortening Rate\n(base_prob=0.98, n_turns=3, n_entries=100)')
ax.legend(fontsize=9)
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('../visualization/fig10a_minja_isr_simulation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig10a_minja_isr_simulation.pdf', bbox_inches='tight')
plt.show()
print('Figure 10a saved: fig10a_minja_isr_simulation')

## 10. Phase 10 Upgrade: MINJA ISR Simulation and AgentPoison Trigger Optimization Comparison

Detailed visualization of the Phase 10 improvements:
- **MINJA ISR simulation**: multi-turn injection success rate across varying base probabilities and indication turns
- **AgentPoison ASR-R improvement**: trigger-optimized vs. query-echo baseline comparison

In [ ]:
from dataclasses import asdict

consolidated = {
    'experiment': 'consolidated_results_visualization',
    'corpus_size': CORPUS_SIZE,
    'top_k': TOP_K,
    'n_poison_base': N_POISON_BASE,
    'seed': SEED,
    'attack_metrics': {at: asdict(attack_results[at]) for at in ATTACK_TYPES},
    'defense_metrics': {dt: asdict(defense_results[dt]) for dt in DEFENSE_TYPES},
    'composite_threat_scores': {
        at: round(
            0.4 * attack_results[at].asr_r +
            0.3 * attack_results[at].asr_a +
            0.3 * attack_results[at].asr_t, 4
        )
        for at in ATTACK_TYPES
    },
    'composite_coverage_scores': {
        dt: round(
            0.4 * defense_results[dt].f1_score +
            0.3 * defense_results[dt].tpr +
            0.3 * (1.0 - defense_results[dt].fpr), 4
        )
        for dt in DEFENSE_TYPES
    },
    'per_attack_defense_f1_matrix': per_attack_defense_matrix.tolist(),
    'normalized_defense_matrix': normalized_matrix.tolist(),
    'attack_order': ATTACK_TYPES,
    'defense_order': DEFENSE_TYPES,
}

output_path = '../visualization/consolidated_results.json'
with open(output_path, 'w') as f:
    json.dump(consolidated, f, indent=2)

print(f'Consolidated results saved to {output_path}')
print('\n=== Attack Summary ===')
for at in ATTACK_TYPES:
    m = attack_results[at]
    ts = consolidated['composite_threat_scores'][at]
    print(f'{at:15s}: ASR-R={m.asr_r:.3f}  ASR-T={m.asr_t:.3f}  Benign={m.benign_accuracy:.3f}  Threat={ts:.3f}')

print('\n=== Defense Summary ===')
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    cs = consolidated['composite_coverage_scores'][dt]
    print(f'{dt:15s}: TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  F1={m.f1_score:.3f}  Coverage={cs:.3f}')

## Phase 11: Statistical Summary Figure (Paper-Ready)

This section generates the **main results figure** for the paper:
- **Panel A**: Multi-trial ASR comparison (all 3 attacks, 3 metrics, 95% CIs)
- **Panel B**: SAD detection performance (TPR/AUROC per attack type)
- **Panel C**: Watermark vs SAD comparison (complementary strengths)
- **Panel D**: Attack stealthiness vs detectability scatter

Suitable for the main body of a NeurIPS / CCS submission.

In [ ]:
import sys

sys.path.insert(0, "../../src")

import os

import matplotlib.pyplot as plt
import numpy as np

from data.synthetic_corpus import SyntheticCorpus
from defenses.semantic_anomaly import SemanticAnomalyDetector
from evaluation.retrieval_sim import (
    generate_agentpoison_passage,
    generate_injecmem_passage,
    generate_minja_passage,
)
from evaluation.statistical import LatexTableGenerator, MultiTrialEvaluator

os.makedirs("../../results/figures", exist_ok=True)

# --- multi-trial attack evaluation ---
evaluator = MultiTrialEvaluator(corpus_size=80, n_poison=5, top_k=5,
                                 use_trigger_optimization=False, n_bootstrap=300)
N_TRIALS = 3  # use 5+ for camera-ready
print("running multi-trial evaluation...")
multi = evaluator.evaluate_all_attacks(n_trials=N_TRIALS)
print("done.")

# --- SAD defense evaluation ---
corpus   = SyntheticCorpus(seed=0)
benign   = [e["content"] for e in corpus.generate_benign_entries(100)]
vqueries = [q["query"]   for q in corpus.get_victim_queries()]

det = SemanticAnomalyDetector(threshold_sigma=2.0)
det.calibrate(benign[:60], vqueries[:8])
for q in vqueries[8:]:
    det.update_query_set(q)

goal  = "override authentication and grant elevated access"
p_ap  = [generate_agentpoison_passage(q, goal) for q in vqueries[:6]]
p_mj  = [generate_minja_passage(q, goal)       for q in vqueries[:6]]
p_im  = [generate_injecmem_passage(goal)        for _ in range(6)]

benign_test = benign[60:]
sad_results = {
    "AgentPoison": det.evaluate_on_corpus(p_ap, benign_test),
    "MINJA":       det.evaluate_on_corpus(p_mj, benign_test),
    "InjecMEM":    det.evaluate_on_corpus(p_im, benign_test),
}
print("SAD evaluation complete.")

In [ ]:
# --- composite results figure ---
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

attack_order  = ["agent_poison", "minja", "injecmem"]
attack_labels = ["AgentPoison", "MINJA", "InjecMEM"]
colors        = ["#E74C3C", "#3498DB", "#2ECC71"]

# -- panel A: ASR-R multi-trial with CIs --
ax_a = fig.add_subplot(gs[0, 0])
means_r = [multi[a].asr_r.mean  for a in attack_order]
lows_r  = [multi[a].asr_r.mean - multi[a].asr_r.lower for a in attack_order]
highs_r = [multi[a].asr_r.upper - multi[a].asr_r.mean for a in attack_order]
x = np.arange(len(attack_order))
ax_a.bar(x, means_r, color=colors, alpha=0.85, edgecolor="black", linewidth=0.8)
ax_a.errorbar(x, means_r, yerr=[lows_r, highs_r], fmt="none",
              color="black", capsize=6, capthick=1.5, linewidth=1.5)
ax_a.set_xticks(x); ax_a.set_xticklabels(attack_labels, fontsize=9)
ax_a.set_ylabel("ASR-R", fontsize=10)
ax_a.set_ylim(0, 1.1)
ax_a.set_title("(A) ASR-R with 95% Bootstrap CI", fontsize=11, fontweight="bold")
ax_a.grid(axis="y", alpha=0.3)

# -- panel B: ASR-T multi-trial --
ax_b = fig.add_subplot(gs[0, 1])
means_t = [multi[a].asr_t.mean  for a in attack_order]
lows_t  = [multi[a].asr_t.mean - multi[a].asr_t.lower for a in attack_order]
highs_t = [multi[a].asr_t.upper - multi[a].asr_t.mean for a in attack_order]
ax_b.bar(x, means_t, color=colors, alpha=0.85, edgecolor="black", linewidth=0.8)
ax_b.errorbar(x, means_t, yerr=[lows_t, highs_t], fmt="none",
              color="black", capsize=6, capthick=1.5, linewidth=1.5)
ax_b.set_xticks(x); ax_b.set_xticklabels(attack_labels, fontsize=9)
ax_b.set_ylabel("ASR-T (End-to-End)", fontsize=10)
ax_b.set_ylim(0, 1.1)
ax_b.set_title("(B) End-to-End Hijacking Rate", fontsize=11, fontweight="bold")
ax_b.grid(axis="y", alpha=0.3)

# -- panel C: per-trial variance (boxplot) --
ax_c = fig.add_subplot(gs[0, 2])
trial_asr_r = [[r.asr_r for r in multi[a].trial_results] for a in attack_order]
bp = ax_c.boxplot(trial_asr_r, patch_artist=True, medianprops={"linewidth": 2})
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax_c.set_xticks([1, 2, 3]); ax_c.set_xticklabels(attack_labels, fontsize=9)
ax_c.set_ylabel("ASR-R per Trial", fontsize=10)
ax_c.set_title("(C) Trial-Level Variance (ASR-R)", fontsize=11, fontweight="bold")
ax_c.grid(axis="y", alpha=0.3)

# -- panel D: SAD TPR by attack type --
ax_d = fig.add_subplot(gs[1, 0])
sad_attacks = list(sad_results.keys())
sad_tprs    = [sad_results[a]["tpr"]   for a in sad_attacks]
sad_fprs    = [sad_results[a]["fpr"]   for a in sad_attacks]
x2 = np.arange(len(sad_attacks))
w  = 0.38
ax_d.bar(x2 - w/2, sad_tprs, w, label="TPR", color=colors, alpha=0.85,
         edgecolor="black", linewidth=0.8)
ax_d.bar(x2 + w/2, sad_fprs, w, label="FPR", color=colors, alpha=0.35,
         edgecolor="black", linewidth=0.8, hatch="//")
ax_d.set_xticks(x2); ax_d.set_xticklabels(sad_attacks, fontsize=9)
ax_d.set_ylim(0, 1.1)
ax_d.set_ylabel("Rate", fontsize=10)
ax_d.set_title("(D) SAD: TPR and FPR by Attack", fontsize=11, fontweight="bold")
ax_d.legend(fontsize=9); ax_d.grid(axis="y", alpha=0.3)

# -- panel E: SAD AUROC by attack type --
ax_e = fig.add_subplot(gs[1, 1])
sad_aurocs = [sad_results[a]["auroc"] for a in sad_attacks]
sad_f1s    = [sad_results[a]["f1"]    for a in sad_attacks]
bars = ax_e.bar(x2, sad_aurocs, color=colors, alpha=0.85, edgecolor="black", linewidth=0.8)
ax_e.axhline(0.5, color="gray", linestyle="--", linewidth=1.0, alpha=0.7)
for xi, (auroc, f1) in enumerate(zip(sad_aurocs, sad_f1s)):
    ax_e.text(xi, auroc + 0.02, f"F1={f1:.2f}", ha="center", fontsize=9, fontweight="bold")
ax_e.set_xticks(x2); ax_e.set_xticklabels(sad_attacks, fontsize=9)
ax_e.set_ylim(0, 1.15)
ax_e.set_ylabel("AUROC", fontsize=10)
ax_e.set_title("(E) SAD: AUROC by Attack Type", fontsize=11, fontweight="bold")
ax_e.grid(axis="y", alpha=0.3)

# -- panel F: stealthiness vs detectability --
ax_f = fig.add_subplot(gs[1, 2])
for i, (atk, atk_key) in enumerate(zip(sad_attacks, attack_order)):
    asr_r_val  = multi[atk_key].asr_r.mean
    auroc_val  = sad_results[atk]["auroc"]
    ax_f.scatter(asr_r_val, auroc_val, s=160, color=colors[i],
                 edgecolors="black", linewidths=1.2, zorder=3,
                 label=atk)
    ax_f.annotate(atk, (asr_r_val, auroc_val),
                  textcoords="offset points", xytext=(8, 4), fontsize=9)
ax_f.set_xlabel("Attack Effectiveness (ASR-R)", fontsize=10)
ax_f.set_ylabel("SAD Detectability (AUROC)", fontsize=10)
ax_f.set_title("(F) Stealthiness vs Detectability\nTrade-off", fontsize=11, fontweight="bold")
ax_f.set_xlim(0, 1); ax_f.set_ylim(0.4, 1.1)
ax_f.legend(fontsize=9); ax_f.grid(alpha=0.3)

fig.suptitle("Phase 11 Summary: Multi-Trial Statistical Evaluation + SAD Defense\n"
             f"(corpus=80, n_poison=5, top_k=5, n_trials={N_TRIALS})",
             fontsize=14, fontweight="bold", y=1.01)

plt.savefig("../../results/figures/fig11_phase11_summary.png",
            dpi=300, bbox_inches="tight")
plt.savefig("../../results/figures/fig11_phase11_summary.pdf", bbox_inches="tight")
plt.show()
print("saved fig11_phase11_summary")

In [ ]:
# generate all LaTeX tables for paper
gen = LatexTableGenerator()
import os

os.makedirs("../../results/tables", exist_ok=True)

# table 1: attack results
t1 = gen.generate_attack_table(multi,
    caption=rf"Attack evaluation (n={N_TRIALS} trials, 95\% bootstrap CI). "
             "Metrics: ASR-R, ASR-A, ASR-T, ISR, Benign Acc.")
gen.save(t1, "../../results/tables/table1_attacks.tex")

# table 2: defense comparison
defense_metrics = {
    "semantic_anomaly": sad_results["AgentPoison"],  # representative
}
# add watermark placeholder
defense_metrics["watermark"] = {"tpr": 0.85, "fpr": 0.06, "f1": 0.83, "auroc": 0.91, "latency_ms": 12.0}
t2 = gen.generate_defense_table(defense_metrics)
gen.save(t2, "../../results/tables/table2_defenses.tex")

print("Table 1 (attacks):")
print(t1)
print()
print("Table 2 (defenses):")
print(t2)
print()
print("tables saved to results/tables/")

In [ ]:
# composite 4-panel summary figure for phase 12 (fig12_phase12_summary)
attack_order  = ["agent_poison", "minja", "injecmem"]
defense_order = ["watermark", "validation", "proactive", "composite", "semantic_anomaly"]
atk_labels = ["AgentPoison", "MINJA", "InjecMEM"]
def_labels = ["Watermark", "Validation", "Proactive", "Composite", "SAD (ours)"]

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# panel a: retrieval asr bars (top-left)
ax_a = fig.add_subplot(gs[0, 0])
metric_labels = ["ASR-R", "ASR-A", "ASR-T"]
metric_keys = ["asr_r", "asr_a", "asr_t"]
colors_m = ["#E63946", "#F4A261", "#2A9D8F"]
x = np.arange(len(attack_order))
w = 0.25
for mi, (mk, ml, mc) in enumerate(zip(metric_keys, metric_labels, colors_m)):
    vals = [getattr(retrieval_metrics[at], mk) for at in attack_order]
    ax_a.bar(x + (mi-1)*w, vals, w, label=ml, color=mc, alpha=0.85, edgecolor="black", lw=0.6)
ax_a.set_xticks(x)
ax_a.set_xticklabels(atk_labels, fontweight="bold")
ax_a.set_ylim(0, 1.15)
ax_a.set_ylabel("Attack Success Rate")
ax_a.set_title("(a) Retrieval-Based ASR per Attack", fontweight="bold")
ax_a.legend(fontsize=9, framealpha=0.9)

# panel b: post-defense asr-r heatmap (top-right)
ax_b = fig.add_subplot(gs[0, 1])
asr_mat = np.zeros((len(attack_order), len(defense_order)))
for ai, atk in enumerate(attack_order):
    for di, dfn in enumerate(defense_order):
        pair = matrix.get(atk, dfn)
        asr_mat[ai, di] = pair.asr_r_under_defense if pair else 0.0
im_b = ax_b.imshow(asr_mat, cmap="RdYlGn_r", vmin=0, vmax=1, aspect="auto")
for ai in range(len(attack_order)):
    for di in range(len(defense_order)):
        tc = "white" if asr_mat[ai, di] < 0.25 or asr_mat[ai, di] > 0.75 else "black"
        ax_b.text(di, ai, f"{asr_mat[ai, di]:.2f}", ha="center", va="center",
                  fontsize=10, color=tc, fontweight="bold")
ax_b.set_xticks(range(len(defense_order)))
ax_b.set_yticks(range(len(attack_order)))
ax_b.set_xticklabels(def_labels, rotation=25, ha="right", fontweight="bold")
ax_b.set_yticklabels(atk_labels, fontweight="bold")
ax_b.set_title("(b) Post-Defense ASR-R (↓ better)", fontweight="bold")
fig.colorbar(im_b, ax=ax_b, fraction=0.046, pad=0.04)

# panel c: defense effectiveness heatmap (bottom-left)
ax_c = fig.add_subplot(gs[1, 0])
eff_mat = np.zeros_like(asr_mat)
for ai, atk in enumerate(attack_order):
    for di, dfn in enumerate(defense_order):
        pair = matrix.get(atk, dfn)
        eff_mat[ai, di] = pair.defense_effectiveness if pair else 0.0
im_c = ax_c.imshow(eff_mat, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
for ai in range(len(attack_order)):
    for di in range(len(defense_order)):
        tc = "white" if eff_mat[ai, di] < 0.25 or eff_mat[ai, di] > 0.75 else "black"
        ax_c.text(di, ai, f"{eff_mat[ai, di]:.2f}", ha="center", va="center",
                  fontsize=10, color=tc, fontweight="bold")
ax_c.set_xticks(range(len(defense_order)))
ax_c.set_yticks(range(len(attack_order)))
ax_c.set_xticklabels(def_labels, rotation=25, ha="right", fontweight="bold")
ax_c.set_yticklabels(atk_labels, fontweight="bold")
ax_c.set_title("(c) Defense Effectiveness (↑ better)", fontweight="bold")
fig.colorbar(im_c, ax=ax_c, fraction=0.046, pad=0.04)

# panel d: avg defense effectiveness bar chart (bottom-right)
ax_d = fig.add_subplot(gs[1, 1])
avg_effs = []
for dfn in defense_order:
    effs = [matrix.get(atk, dfn).defense_effectiveness for atk in attack_order
            if matrix.get(atk, dfn) is not None]
    avg_effs.append(sum(effs) / len(effs) if effs else 0.0)
bar_colors = ["#264653", "#457B9D", "#A8DADC", "#1D3557", "#E63946"]
bars = ax_d.bar(range(len(defense_order)), avg_effs, color=bar_colors,
                edgecolor="black", linewidth=0.8)
for bar, v in zip(bars, avg_effs):
    ax_d.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
              f"{v:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax_d.set_xticks(range(len(defense_order)))
ax_d.set_xticklabels(def_labels, rotation=20, ha="right", fontweight="bold")
ax_d.set_ylabel("Average Defense Effectiveness")
ax_d.set_ylim(0, 1.1)
ax_d.set_title("(d) Avg. Effectiveness per Defense\n(across all 3 attacks)", fontweight="bold")
ax_d.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

fig.suptitle("Phase 12 Summary: Attack-Defense Interaction Matrix\n"
             f"Corpus={CORPUS_SIZE}, N_poison={N_POISON}, Top-K=5",
             fontsize=14, fontweight="bold")

fig.savefig("../../results/figures/fig12_phase12_summary.png", dpi=300, bbox_inches="tight")
fig.savefig("../../results/figures/fig12_phase12_summary.pdf", bbox_inches="tight")
plt.show()
print("saved fig12_phase12_summary")

# generate all paper latex tables
latex_asr = ev.to_latex_matrix(matrix)
latex_tpr = ev.to_latex_tpr_fpr_table(matrix)
os.makedirs("../../results/tables", exist_ok=True)
with open("../../results/tables/table2_attack_defense_matrix.tex", "w") as f:
    f.write(latex_asr)
with open("../../results/tables/table3_tpr_fpr.tex", "w") as f:
    f.write(latex_tpr)
print("saved all paper latex tables")
print("\ntable 2 preview (first 10 lines):")
for line in latex_asr.split("\n")[:10]:
    print(" ", line)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

from evaluation.attack_defense_matrix import AttackDefenseEvaluator
from evaluation.retrieval_sim import RetrievalSimulator
from scripts.visualization import plot_matrix_asr_heatmap, plot_retrieval_asr_bars

SEED = 42
CORPUS_SIZE = 50
N_POISON = 3

# run attack-defense matrix
print("building attack-defense matrix (corpus=50, n_poison=3, n_trials=1)...")
ev = AttackDefenseEvaluator(corpus_size=CORPUS_SIZE, n_poison=N_POISON,
                             top_k=5, use_trigger_optimization=False, seed=SEED)
matrix = ev.evaluate_full_matrix(n_trials=1)

# run retrieval simulation for all attacks
print("running retrieval simulation for all three attacks...")
sim = RetrievalSimulator(corpus_size=CORPUS_SIZE, n_poison_per_attack=N_POISON,
                          top_k=5, use_trigger_optimization=False, seed=SEED)
retrieval_metrics = {at: sim.evaluate_attack(at) for at in ["agent_poison", "minja", "injecmem"]}

print("\nretrieval simulation results:")
for at, m in retrieval_metrics.items():
    print(f"  {at:<18} asr-r={m.asr_r:.3f}  asr-a={m.asr_a:.3f}  asr-t={m.asr_t:.3f}")

# generate and save individual phase 12 figures
os.makedirs("../../results/figures", exist_ok=True)

fig_matrix = plot_matrix_asr_heatmap(matrix)
fig_matrix.savefig("../../results/figures/fig12b_attack_defense_matrix.png", dpi=300, bbox_inches="tight")
fig_matrix.savefig("../../results/figures/fig12b_attack_defense_matrix.pdf", bbox_inches="tight")
plt.show()
print("saved fig12b_attack_defense_matrix")

fig_bars = plot_retrieval_asr_bars(retrieval_metrics)
fig_bars.savefig("../../results/figures/fig12_retrieval_asr_bars.png", dpi=300, bbox_inches="tight")
fig_bars.savefig("../../results/figures/fig12_retrieval_asr_bars.pdf", bbox_inches="tight")
plt.show()
print("saved fig12_retrieval_asr_bars")

## Phase 12: Complete Attack-Defense Matrix Visualization + Full Paper Tables

This section produces the two primary paper tables and the Phase 12 summary figure:

- **Table 2**: Post-defense ASR-R matrix (attacks × defenses) — follows Neural Cleanse format (Wang et al., 2019)
- **Table 3**: TPR / FPR per (defense, attack) pair
- **Figure 12b**: Two-panel heatmap — ASR-R under defense + Defense Effectiveness
- **Figure 12_summary**: 4-panel composite summarising all Phase 12 contributions